# Analisis Frekuensi Area Terbakar Berulang & Konsistensi Spasial (Landsat vs. MODIS)
## Studi Kasus: Kabupaten Pulang Pisau, Kalimantan Tengah (Periode 2001–2024)

**Mata Kuliah:** Pengolahan Citra Digital (PCD) — Semester 5  
**Institusi:** D4 Teknik Informatika, Politeknik Negeri Bandung (POLBAN)  
**Kelompok:**  
1. **Roufiel Hadi** (241524028)  
2. **Amalia Khoerina** (241524003)  

---
### 📌 Deskripsi Proyek
Notebook ini memproses 4 raster GeoTIFF hasil ekstraksi Google Earth Engine (GEE) pada **Grid Analisis Terpadu 250m (Opsi B)**:
1. `Frekuensi_Terbakar_Landsat_2001_2024.tif`: Akumulasi frekuensi piksel terbakar dari produk berbasis Landsat (30m).
2. `Frekuensi_Terbakar_MODIS_2001_2024.tif`: Akumulasi frekuensi piksel terbakar dari produk MODIS MCD64A1 (500m).
3. `Spatial_Agreement_Landsat_MODIS_Pulang_Pisau.tif`: Peta kesepakatan spasial 4 kelas (Sepakat, Landsat-only, MODIS-only, Tidak terbakar).
4. `Kronologi_Deforestasi_Hansen_Pulang_Pisau.tif`: Data kronologi tahun kehilangan tutupan kanopi pohon (Hansen GFC 2001–2023).

---
### 🚀 Cara Menjalankan di Google Colab
1. Jalankan **Cell 1** untuk menginstal dependensi.
2. Jalankan **Cell 2** untuk memilih sumber dataset (bisa upload manual 4 file `.tif` ke folder `Dataset/` atau mount Google Drive).

In [ ]:
# Cell 1: Instalasi & Import Library PCD
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

!pip install -q opencv-python matplotlib seaborn numpy pandas

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

print("[*] Environment & library PCD siap digunakan!")

In [ ]:
# Cell 2: Konfigurasi & Deteksi Lokasi Dataset
def locate_dataset():
    candidates = [
        'Dataset',
        '../Dataset',
        './Dataset',
        '/content/Dataset',
        '/content/drive/MyDrive/Tubes_PCD_Pulang_Pisau',
        os.path.join(os.getcwd(), 'Dataset'),
        os.path.join(os.path.dirname(os.getcwd()), 'Dataset')
    ]
    for c in candidates:
        if os.path.exists(c) and any(f.endswith('.tif') for f in os.listdir(c)):
            return os.path.abspath(c)
    return None

dataset_dir = locate_dataset()

if dataset_dir is None:
    print("[!] Direktori Dataset belum terdeteksi secara otomatis.")
    print("    Silakan buat folder 'Dataset' di panel files sebelah kiri dan unggah 4 file .tif hasil GEE.")
    os.makedirs('Dataset', exist_ok=True)
    dataset_dir = os.path.abspath('Dataset')
else:
    print(f"[✔] Direktori Dataset terdeteksi: {dataset_dir}")

In [ ]:
# Cell 3: Memuat Raster GeoTIFF & Validasi Dimensi
file_mapping = {
    'landsat': 'Frekuensi_Terbakar_Landsat_2001_2024.tif',
    'modis': 'Frekuensi_Terbakar_MODIS_2001_2024.tif',
    'agreement': 'Spatial_Agreement_Landsat_MODIS_Pulang_Pisau.tif',
    'deforest': 'Kronologi_Deforestasi_Hansen_Pulang_Pisau.tif'
}

rasters = {}
info_rows = []
PIXEL_HA = (250.0 * 250.0) / 10000.0  # 6.25 ha per sel 250m

for key, fname in file_mapping.items():
    p = os.path.join(dataset_dir, fname)
    if not os.path.exists(p):
        raise FileNotFoundError(f"File tidak ditemukan: {p}. Pastikan file telah diunggah.")
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    rasters[key] = img
    info_rows.append({
        'Nama Raster': fname,
        'Tinggi (px)': img.shape[0],
        'Lebar (px)': img.shape[1],
        'Tipe Data': str(img.dtype),
        'Nilai Min': int(np.min(img)),
        'Nilai Max': int(np.max(img))
    })

df_info = pd.DataFrame(info_rows)
print("=== KARAKTERISTIK DATASET GEOTIFF PCD ===")
display(df_info)

--- 
### 🔬 1. Evaluasi Konsistensi Spasial Multi-Sensor (IoU & Dice Coefficient)

Didefinisikan himpunan biner piksel terbakar ($F \ge 1$):
* $A$ = Mask piksel terbakar Landsat
* $B$ = Mask piksel terbakar MODIS

$$\text{IoU (Jaccard Index)} = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Intersection}}{\text{Union}}$$

$$\text{Dice Similarity Coefficient} = \frac{2 \times |A \cap B|}{|A| + |B|}$$

In [ ]:
# Cell 4: Perhitungan Metrik IoU, Dice, dan Matriks Kesepakatan Spasial
f_landsat = rasters['landsat']
f_modis = rasters['modis']
agreement = rasters['agreement']
deforest = rasters['deforest']

l_burned = f_landsat >= 1
m_burned = f_modis >= 1

intersection = np.logical_and(l_burned, m_burned).sum()
union = np.logical_or(l_burned, m_burned).sum()
iou = intersection / union if union > 0 else 0.0
dice = (2.0 * intersection) / (l_burned.sum() + m_burned.sum())

print("=" * 60)
print("HASIL EVALUASI METRIK KONSISTENSI SPASIAL (2001–2024)")
print("=" * 60)
print(f"Total Piksel Landsat Terbakar : {l_burned.sum():,} piksel ({l_burned.sum() * PIXEL_HA:,.1f} ha)")
print(f"Total Piksel MODIS Terbakar   : {m_burned.sum():,} piksel ({m_burned.sum() * PIXEL_HA:,.1f} ha)")
print(f"Irisan (Mutual Agreement)     : {intersection:,} piksel ({intersection * PIXEL_HA:,.1f} ha)")
print(f"Gabungan (Union)              : {union:,} piksel ({union * PIXEL_HA:,.1f} ha)")
print(f"-> IoU (Jaccard Index)        : {iou:.4f} ({iou * 100:.2f}%)")
print(f"-> Dice Similarity Coefficient: {dice:.4f} ({dice * 100:.2f}%)")

# Tabel Matriks Kesepakatan Spasial
agr_labels = ['Keduanya Tidak Terbakar', 'Hanya Landsat Terbakar', 'Hanya MODIS Terbakar', 'Keduanya Sepakat Terbakar']
df_agr = pd.DataFrame({
    'Kategori Kesepakatan': agr_labels,
    'Piksel': [np.sum(agreement == i) for i in range(4)],
    'Luas (ha)': [np.sum(agreement == i) * PIXEL_HA for i in range(4)],
    'Persentase Wilayah (%)': [(np.sum(agreement == i) / agreement.size) * 100 for i in range(4)]
})

display(df_agr)

--- 
### 📊 2. Analisis Distribusi Frekuensi Terbakar Berulang ($F(x,y)$)

Klasifikasi jumlah tahun terbakar selama rentang 24 tahun (2001–2024):
* **Kelas 0:** 0 Tahun (Tidak pernah terbakar)
* **Kelas 1:** 1 Tahun (Kebakaran insidental/tunggal)
* **Kelas 2:** 2–4 Tahun (Frekuensi rendah–sedang)
* **Kelas 3:** 5–8 Tahun (Frekuensi sedang–tinggi)
* **Kelas 4:** $\ge 9$ Tahun (Frekuensi sangat tinggi / titik api kronis)

In [ ]:
# Cell 5: Klasifikasi dan Visualisasi Distribusi Frekuensi
def classify_freq(arr):
    cls = np.zeros_like(arr, dtype=np.uint8)
    cls[arr == 1] = 1
    cls[(arr >= 2) & (arr <= 4)] = 2
    cls[(arr >= 5) & (arr <= 8)] = 3
    cls[arr >= 9] = 4
    return cls

cls_l = classify_freq(f_landsat)
cls_m = classify_freq(f_modis)

class_names = ["0 Tahun (Tidak)", "1 Tahun (Insidental)", "2–4 Tahun (Rendah–Sedang)", "5–8 Tahun (Sedang–Tinggi)", "≥ 9 Tahun (Kronis)"]
df_freq = pd.DataFrame({
    'Kelas Frekuensi': class_names,
    'Luas Landsat (ha)': [np.sum(cls_l == i) * PIXEL_HA for i in range(5)],
    'Luas MODIS (ha)': [np.sum(cls_m == i) * PIXEL_HA for i in range(5)]
})

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
x = np.arange(len(class_names))
w = 0.35
ax.bar(x - w/2, df_freq['Luas Landsat (ha)'] / 1000.0, width=w, label='Landsat (30m)', color='#d73027', alpha=0.9)
ax.bar(x + w/2, df_freq['Luas MODIS (ha)'] / 1000.0, width=w, label='MODIS (500m)', color='#fdae61', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(class_names, fontsize=10)
ax.set_ylabel('Luas Area (Ribu Hektar)', fontsize=11, fontweight='bold')
ax.set_title('Distribusi Luas Wilayah Berdasarkan Kelas Frekuensi Terbakar di Pulang Pisau', fontsize=12, fontweight='bold')
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

--- 
### 🌲 3. Analisis Korelasi dengan Deforestasi Hansen GFC (Pelengkap)

Menghitung berapa banyak kehilangan tutupan kanopi pohon yang dipicu oleh kebakaran hutan berulang ($F \ge 2$).

In [ ]:
# Cell 6: Evaluasi Deforestasi & Tren Tahunan (2001–2023)
deforested = deforest >= 1
recurrent_fire = np.logical_or(f_landsat >= 2, f_modis >= 2)
defor_in_fire = np.logical_and(deforested, recurrent_fire).sum()
pct_defor_fire = (defor_in_fire / deforested.sum()) * 100

print("=" * 60)
print("KORELASI KARHUTLA DENGAN DEFORESTASI HANSEN GFC")
print("=" * 60)
print(f"Total Luas Deforestasi (2001–2023)    : {deforested.sum():,} piksel ({deforested.sum() * PIXEL_HA:,.1f} ha)")
print(f"Deforestasi di Area Kebakaran Berulang: {defor_in_fire:,} piksel ({defor_in_fire * PIXEL_HA:,.1f} ha)")
print(f"-> {pct_defor_fire:.2f}% deforestasi terjadi di zona kebakaran berulang!")

# Grafik Tren Kehilangan Hutan per Tahun
years = np.arange(2001, 2024)
loss_ha = [np.sum(deforest == (y - 2000)) * PIXEL_HA for y in years]

fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
ax.bar(years, np.array(loss_ha) / 1000.0, color='#b2182b', alpha=0.85, edgecolor='#67001f')

for y in [2002, 2006, 2009, 2014, 2015, 2019, 2023]:
    idx = y - 2001
    ax.annotate('El Niño', (y, loss_ha[idx] / 1000.0 + 3), ha='center', fontsize=8, fontweight='bold', color='#67001f', rotation=45)

ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, fontsize=9)
ax.set_ylabel('Luas Deforestasi (Ribu Hektar)', fontsize=11, fontweight='bold')
ax.set_title('Kronologi Kehilangan Hutan Tahunan di Kabupaten Pulang Pisau (Hansen GFC 2001–2023)', fontsize=12, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

--- 
### 🗺️ 4. Visualisasi Geospasial 4-Panel (Publikasi Akademik)

In [ ]:
# Cell 7: Render Peta 4-Panel Geospasial
fig, axes = plt.subplots(2, 2, figsize=(16, 18), dpi=150)
plt.subplots_adjust(wspace=0.15, hspace=0.15)

# Panel A: Frekuensi Landsat
cmap_freq = mcolors.ListedColormap(['#2b83ba', '#ffffbf', '#fdae61', '#f46d43', '#d73027', '#67001f'])
bounds_freq = [0, 1, 2, 5, 9, 15, 25]
norm_freq = mcolors.BoundaryNorm(bounds_freq, cmap_freq.N)

im0 = axes[0, 0].imshow(f_landsat, cmap=cmap_freq, norm=norm_freq)
axes[0, 0].set_title('(A) Frekuensi Terbakar Berulang Landsat (30m / 250m)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')
cbar0 = plt.colorbar(im0, ax=axes[0, 0], orientation='horizontal', pad=0.04, fraction=0.046)
cbar0.set_label('Jumlah Tahun Terbakar (2001–2024)', fontsize=10)

# Panel B: Frekuensi MODIS
im1 = axes[0, 1].imshow(f_modis, cmap=cmap_freq, norm=norm_freq)
axes[0, 1].set_title('(B) Frekuensi Terbakar Berulang MODIS (500m / 250m)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')
cbar1 = plt.colorbar(im1, ax=axes[0, 1], orientation='horizontal', pad=0.04, fraction=0.046)
cbar1.set_label('Jumlah Tahun Terbakar (2001–2024)', fontsize=10)

# Panel C: Spatial Agreement
cmap_agr = mcolors.ListedColormap(['#252525', '#fee08b', '#fdae61', '#d73027'])
bounds_agr = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm_agr = mcolors.BoundaryNorm(bounds_agr, cmap_agr.N)

im2 = axes[1, 0].imshow(agreement, cmap=cmap_agr, norm=norm_agr)
axes[1, 0].set_title(f'(C) Peta Kesepakatan Spasial (IoU: {iou:.2%}, Dice: {dice:.2%})', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')
cbar2 = plt.colorbar(im2, ax=axes[1, 0], orientation='horizontal', pad=0.04, fraction=0.046, ticks=[0, 1, 2, 3])
cbar2.ax.set_xticklabels(['Tidak', 'Landsat Saja', 'MODIS Saja', 'Sepakat'], fontsize=9)

# Panel D: Kronologi Deforestasi Hansen
cmap_def = plt.colormaps['YlOrRd'].resampled(23)
cmap_def.set_under('#252525')
im3 = axes[1, 1].imshow(deforest, cmap=cmap_def, vmin=1, vmax=23)
axes[1, 1].set_title('(D) Kronologi Deforestasi Kanopi Pohon (Hansen GFC 2001–2023)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')
cbar3 = plt.colorbar(im3, ax=axes[1, 1], orientation='horizontal', pad=0.04, fraction=0.046)
cbar3.set_label('Tahun Kehilangan Kanopi (1=2001, 15=2015, 23=2023)', fontsize=10)

plt.show()
print("[✔] Seluruh analisis spasial dan visualisasi berhasil dieksekusi!")